In [132]:
import scipy.signal as sig
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig
import matplotlib.pyplot as plt
import scipy.io as sio
import scipy.interpolate as spi


In [163]:
sio.whosmat("ECG_TP4.mat")
mat_struct = sio.loadmat("ECG_TP4.mat")

ecg_one_lead = mat_struct["ecg_lead"]
ecg_one_lead = ecg_one_lead.flatten()

qrs_detections = mat_struct["qrs_detections"]


In [164]:
fs = 1000  # Hz
nyq_frec = fs / 2

N = 10000

In [150]:
ecg_one_lead = ecg_one_lead[:N]
df_p = fs / N

In [151]:
frequencies = np.fft.fftfreq(N, 1/fs)  # frequency axis
fft_values = np.fft.fft(ecg_one_lead)      # Fourier Transform

# Calculate the magnitude spectrum in dB
magnitude_spectrum_db = 20 * np.log10(np.abs(fft_values) / N)

# Plotting the frequency spectrum in dB (only positive frequencies)
plt.figure(figsize=(10, 6))
plt.plot(frequencies[:N // 2], magnitude_spectrum_db[:N // 2])  # positive frequencies
plt.title('Frequency Spectrum of ECG Signal (in dB)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.grid()
plt.show()

In [152]:
ms = 75
s = ms / 1000
offset = int(s * fs)
offset

75

In [153]:
from itertools import takewhile

slice_of_detections = list(takewhile(lambda x: x <= N, qrs_detections))
print(max(slice_of_detections))

[1128482]


In [154]:
puntos_de_interes = [int(m[0]) - offset for m in slice_of_detections]

In [155]:
plt.figure(figsize=(10, 6))
plt.plot(ecg_one_lead, label='ECG Signal')

plt.plot(puntos_de_interes, ecg_one_lead[puntos_de_interes], 'x', label='Points of Interest')

# Adding labels and legend
plt.title('ECG Signal with Points of Interest')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.legend()
plt.grid()
plt.show()

In [156]:
amplitudes_of_interest = ecg_one_lead[puntos_de_interes]

In [157]:
cs = spi.CubicSpline(puntos_de_interes, amplitudes_of_interest)

In [158]:
x = np.linspace(0, N -1, N)
noise_interpolated = cs(x)

In [160]:
plt.figure(figsize=(10, 5))
plt.plot(ecg_one_lead, label="ECG Lead", color="blue")
plt.plot(x, noise_interpolated, label="Cubic Spline Interpolation", color="red")
#plt.scatter(puntos_de_interes, amplitudes_of_interest, color="green", label="Points of Interest", zorder=5, marker='x')
plt.legend()
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("ECG Lead with Cubic Spline Interpolation of Points of Interest")
plt.show()

In [161]:
filtered_signal = ecg_one_lead - noise_interpolated

In [165]:
plt.figure(figsize=(10, 5))
plt.plot(ecg_one_lead, label="ECG Lead", color="blue")
plt.plot(filtered_signal, label="Cubic Spline Interpolation", color="red")
#plt.scatter(puntos_de_interes, amplitudes_of_interest, color="green", label="Points of Interest", zorder=5, marker='x')
plt.legend()
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("ECG Lead with Cubic Spline Interpolation of Points of Interest")
plt.show()

In [162]:
frequencies = np.fft.fftfreq(N, 1/fs)  # frequency axis
fft_values = np.fft.fft(filtered_signal)      # Fourier Transform

# Calculate the magnitude spectrum in dB
filtered_magnitude_spectrum_db = 20 * np.log10(np.abs(fft_values) / N)

# Plotting the frequency spectrum in dB (only positive frequencies)
plt.figure(figsize=(10, 6))
plt.plot(frequencies[:N // 2], filtered_magnitude_spectrum_db[:N // 2], label='Filtered Signal', color='blue')
plt.plot(frequencies[:N // 2], magnitude_spectrum_db[:N // 2], label='Original Signal', color='red', linestyle='--')
plt.title('Frequency Spectrum of ECG Signal (in dB)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.grid()
plt.legend()
plt.show()

In [ ]:
filtered_median_signal = sig.medfilt(ecg_one_lead, )